# Logistic regression — stable implementation and a leakage-safe baseline

This notebook explains the logistic-regression slice of the CSC 781 showcase and
reads the evidence the module generated. The reusable implementation lives in
[`mlshowcase/logistic.py`](../mlshowcase/logistic.py); the narrative write-up is
[`docs/logistic.md`](../docs/logistic.md) and the generated evidence is
[`results/logistic.json`](../results/logistic.json).

The protocol, rebuilt from Module 6 Assignment 6:

- **Data.** scikit-learn's bundled breast-cancer dataset (no download). The
  source encodes malignant as 0 and benign as 1; this project remaps
  **malignant to positive class 1** and benign to 0, so precision, recall and
  F1 always describe the clinically interesting class.
- **Splits.** One stratified 60 / 20 / 20 train / development / test split per
  seed (42, 43, 44) from the shared `stratified_split` utility.
- **Preprocessing.** The z-score scaler is fitted on the training split only;
  the same stored statistics transform development and test.
- **Optimization.** Handwritten full-batch gradient descent on the mean binary
  cross-entropy for learning rates 0.01, 0.05, 0.1 and 0.5, capped at 3000
  steps with an explicit convergence policy.
- **Selection.** The learning rate and decision threshold are chosen jointly by
  development positive-class F1; the test split is scored once, after both
  choices are fixed.
- **Comparison.** An unregularized `LogisticRegression(C=np.inf)` baseline
  uses the same preprocessing and its own independent development threshold.
- **Scope.** A methods showcase on a small, public dataset: **no clinical
  deployment claim**.

The notebook never retrains the full experiment. It illustrates the reusable
API on small synthetic inputs and then reads the stored JSON and figures.


## Why the implementation is numerically stable

The historical notebook evaluated $\sigma(z) = 1 / (1 + e^{-z})$ directly,
which overflows `exp` for large negative scores. The reusable implementation
uses the algebraically identical two-branch form

$$
\sigma(z) =
\begin{cases}
1 / (1 + e^{-z}), & z \ge 0, \\
e^{z} / (1 + e^{z}), & z < 0,
\end{cases}
$$

so `exp` always receives a non-positive argument. Nothing is hidden behind
warning filters: the implementation cannot overflow in the first place.

Cross-entropy is computed from **logits** with `numpy.logaddexp`. For a linear
score $z_i = \theta_0 + x_i^\top \theta_{1:}$ and label $y_i \in \{0, 1\}$,

$$
J(\theta) = \frac{1}{m} \sum_{i=1}^{m}
\begin{cases}
\mathrm{logaddexp}(0, -z_i), & y_i = 1, \\
\mathrm{logaddexp}(0, z_i), & y_i = 0,
\end{cases}
$$

which equals the usual cross-entropy but stays finite for large $|z_i|$. Its
gradient,

$$
\nabla_\theta J = \frac{1}{m} X^\top \left( \sigma(X\theta) - y \right),
$$

is exposed as `logistic_gradient`, while `logistic_loss` exposes the objective,
so the two can be compared with central finite differences.


In [ ]:
import json
import os
import sys
import tomllib
from pathlib import Path

# This notebook runs with its checkout's ``notebooks/`` directory as the working
# directory, so ``..`` is the public checkout root. Set MLSHOWCASE_ROOT to the
# checkout's absolute path to start the notebook from anywhere else.
EXPECTED_PROJECT = "mlshowcase"


def require_showcase_checkout(root: Path, *required_files: str) -> None:
    """Fail before any import or result read unless ``root`` is this checkout."""
    guide = (
        f"{root} is not the mlshowcase checkout this notebook belongs to: run the "
        "notebook with its notebooks/ directory as the working directory, or set "
        "MLSHOWCASE_ROOT to the checkout's absolute path"
    )
    pyproject = root / "pyproject.toml"
    if not pyproject.is_file():
        raise FileNotFoundError(f"{guide} (no pyproject.toml found there)")
    try:
        name = tomllib.loads(pyproject.read_text(encoding="utf-8"))["project"]["name"]
    except (KeyError, tomllib.TOMLDecodeError) as error:
        raise ValueError(f"{guide} (project.name could not be read)") from error
    if name != EXPECTED_PROJECT:
        raise ValueError(f"{guide} (project.name is {name!r})")
    for relative in required_files:
        if not (root / relative).is_file():
            raise FileNotFoundError(f"{guide} (missing {relative})")


_override = os.environ.get("MLSHOWCASE_ROOT")
if _override and not Path(_override).is_absolute():
    raise ValueError(f"MLSHOWCASE_ROOT must be an absolute path, not {_override!r}")
SHOWCASE_ROOT = Path(_override).resolve() if _override else Path("..").resolve()
require_showcase_checkout(SHOWCASE_ROOT, "mlshowcase/logistic.py", "notebooks/logistic.ipynb")

if str(SHOWCASE_ROOT) not in sys.path:
    sys.path.insert(0, str(SHOWCASE_ROOT))

import numpy as np  # noqa: E402

from mlshowcase.logistic import (  # noqa: E402
    FeatureScaler,
    add_intercept,
    cross_entropy,
    fit,
    logistic_gradient,
    logistic_loss,
    predict,
    predict_proba,
    sigmoid,
)

print("showcase checkout:", SHOWCASE_ROOT.name)


## Stability in practice

Extreme scores and logits are exactly the cases that break the original
`exp`-based formula. Neither call below may raise an overflow warning, and both
stay finite.


In [ ]:
extreme_scores = np.array([-1000.0, -40.0, 0.0, 40.0, 1000.0])
print("sigmoid:", sigmoid(extreme_scores))
print(
    "cross-entropy at extreme logits:",
    cross_entropy(np.array([0.0, 1.0]), np.array([1000.0, -1000.0])),
)

rng = np.random.default_rng(0)
design = add_intercept(rng.normal(size=(6, 2)))
labels = np.array([0.0, 1.0, 0.0, 1.0, 1.0, 0.0])
theta = np.array([0.2, -0.4, 0.6])
epsilon = 1e-6
numeric = np.array(
    [
        (
            logistic_loss(design, labels, theta + epsilon * step)
            - logistic_loss(design, labels, theta - epsilon * step)
        )
        / (2.0 * epsilon)
        for step in np.eye(theta.size)
    ]
)
print("analytic gradient:", logistic_gradient(design, labels, theta))
print("finite differences:", numeric)


## Train-only scaling

The historical `normalize` function computed separate means and standard
deviations for the train, development and test arrays, which leaks held-out
information into preprocessing. `FeatureScaler` fits once, stores `mean` and
`scale`, and transforms later data with those stored values. A constant column
has zero standard deviation, so its stored scale stays 1.0 and it transforms to
exact zeros instead of `NaN`.


In [ ]:
training = np.array([[1.0, 5.0], [1.0, 7.0], [1.0, 9.0]])
scaler = FeatureScaler.fit(training)
print("stored mean:", scaler.mean)
print("stored scale:", scaler.scale, "(constant column keeps scale 1.0)")
print("later rows scaled with the training statistics:")
print(scaler.transform(np.array([[1.0, 11.0], [1.0, -3.0]])))


## A tiny fit, for illustration only

`fit` runs handwritten full-batch gradient descent and returns a frozen
`FitResult` with the parameters and the optimization history. This synthetic
two-cloud problem shows the API; the experiment numbers in the next section
come from the generated JSON, not from this cell.


In [ ]:
rng = np.random.default_rng(4)
negative = rng.normal(loc=-1.5, scale=1.0, size=(40, 2))
positive = rng.normal(loc=1.5, scale=1.0, size=(40, 2))
demo_features = np.vstack((negative, positive))
demo_labels = np.concatenate((np.zeros(40), np.ones(40)))

result = fit(demo_features, demo_labels, learning_rate=0.5, max_steps=800, tolerance=1e-7)
print(
    f"converged={result.converged} steps={result.steps} "
    f"final loss={result.loss_history[-1]:.6f}"
)
probabilities = predict_proba(demo_features, result.theta)
print("first five probabilities:", np.round(probabilities[:5], 4))
for threshold in (0.1, 0.5, 0.9):
    positives = int(predict(demo_features, result.theta, threshold).sum())
    print(f"threshold {threshold}: {positives} positive predictions of 80")


## Generated evidence

The full experiment runs only through the guarded command line entry point:

```console
uv run --locked python -m mlshowcase.logistic --output-dir results
```

It writes `results/logistic.json` — protocol, per-seed selections, held-out
metrics, optimization curves and fit/selection/evaluation timings — plus
`results/logistic_loss.png` and `results/logistic_comparison.png`. The cells
below read those artifacts and never retrain the three-seed protocol; if the
JSON is missing they raise a helpful error instead of falling back to example
data. The write-up lives in [`docs/logistic.md`](../docs/logistic.md).


In [ ]:
results_path = SHOWCASE_ROOT / "results" / "logistic.json"
if not results_path.is_file():
    raise FileNotFoundError(
        f"Missing {results_path.relative_to(SHOWCASE_ROOT)}. Run "
        "`uv run --locked python -m mlshowcase.logistic --output-dir results` from the "
        "repository root first."
    )

results = json.loads(results_path.read_text(encoding="utf-8"))
print("seeds:", results["protocol"]["splits"]["seeds"])
print("full class counts:", results["protocol"]["provenance"]["class_counts"])
print("label remap:", results["protocol"]["provenance"]["label_remap"])
print("environment:", results["environment"])


## Selections and held-out metrics

Selection uses the development split only; the test split is scored once, after
the learning rate and threshold are fixed. Both the handwritten model and the
baseline choose their own threshold on development, so the comparison stays
honest.


In [ ]:
for run in results["runs"]:
    selection = run["selection"]
    final = run["final"]
    baseline = run["baseline"]["final"]
    print(
        f"seed {run['seed']}: rate={selection['learning_rate']} "
        f"threshold={selection['threshold']} dev F1={selection['dev_f1']:.3f}"
    )
    print(
        "  custom   "
        f"accuracy={final['accuracy']:.3f} precision={final['positive_precision']:.3f} "
        f"recall={final['positive_recall']:.3f} F1={final['positive_f1']:.3f} "
        f"AUC={final['roc_auc']:.3f}"
    )
    print(
        "  baseline "
        f"accuracy={baseline['accuracy']:.3f} precision={baseline['positive_precision']:.3f} "
        f"recall={baseline['positive_recall']:.3f} F1={baseline['positive_f1']:.3f} "
        f"AUC={baseline['roc_auc']:.3f}"
    )


In [ ]:
summary = results["summary"]
for name in ("custom", "baseline"):
    print(
        name,
        {metric: round(values["mean"], 3) for metric, values in summary[name].items()},
    )
print("selection per seed:", summary["selections"])


In [ ]:
for run in results["runs"]:
    for rate in run["rates"]:
        print(
            f"seed {run['seed']} rate {rate['learning_rate']}: "
            f"steps={rate['steps']} converged={rate['converged']} "
            f"diverged={rate['diverged']} final loss={rate['final_loss']:.4f}"
        )
for run in results["runs"]:
    print(f"seed {run['seed']} timings (ms): {run['timings_ms']}")


## Figures

`logistic_loss.png` shows the training loss against the gradient-descent step
for every learning rate and seed: stable rates decrease smoothly, while an
unstable rate is visible as a plateau or growth. `logistic_comparison.png`
shows the development F1 threshold curves with the selected points, and the
held-out metrics of the selected model next to the unregularized baseline.


In [ ]:
from IPython.display import Image, display

for figure_name in ("logistic_loss.png", "logistic_comparison.png"):
    figure_path = SHOWCASE_ROOT / "results" / figure_name
    if not figure_path.is_file():
        raise FileNotFoundError(
            f"Missing {figure_path.relative_to(SHOWCASE_ROOT)}; run the experiment "
            "command above first."
        )
    display(Image(filename=str(figure_path)))


## What the rebuild corrects

- **Overflowing sigmoid.** The original `1 / (1 + np.exp(-z))` is replaced by
  the stable two-branch form, with no warning suppression anywhere.
- **Leaky normalization.** The original scaled train, development and test
  independently; `FeatureScaler` stores training statistics once and applies
  them everywhere.
- **Last model instead of the selected one.** The original printed evaluations
  from whatever `theta` was fitted last. Here the selected learning rate's
  parameters are the only ones evaluated, and only after the threshold is
  chosen.
- **Undefined positive class.** The original used scikit-learn's default, which
  made benign the positive class. Malignant is now positive class 1, recorded
  explicitly in the protocol.
- **Unstated convergence policy.** The rebuild documents the gradient-norm
  tolerance, the step cap, and the divergence handling that excludes a
  non-finite run from selection.
- **No error bars.** Every configuration is repeated for three seeds, and the
  JSON reports the repeat summary.


## Limitations

- The bundled dataset is small and comes from a single source; these numbers
  describe this protocol on this dataset only, and no clinical deployment
  claim is made.
- The model is unregularized linear logistic regression, so it cannot express
  interactions or nonlinear boundaries.
- Selection maximizes positive-class F1 on development; a different criterion
  (for example recall at a fixed precision) would select different thresholds.
- Timings are single-machine wall-clock observations, not benchmarks.


## Reproducing

```console
uv run --locked python -m mlshowcase.logistic --output-dir results
uv run --locked pytest tests/test_logistic.py
```

The experiment stores no absolute paths, hostnames or usernames; the JSON
environment block is limited to public-safe interpreter, OS, CPU and package
versions.
